In [52]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [53]:
# read in all words
with open('names.txt', 'r', encoding='UTF-8') as f:
    words = f.read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [54]:
len(words)

32033

In [55]:
# build the vocabulary of the characters and mappings to/from integers
chars = ['.'] + sorted(list(set(''.join(words))))
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}
print(itos)

{0: '.', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z'}


In [87]:
# Set default device at the start of your script
if torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.set_default_device(device)
else:
    device = torch.device("cpu")
    torch.set_default_device(device)
print(f'Using device: {device}')

Using device: mps


In [57]:
# The bigram model could produce a new character with only one character
# of context, leading to a fairly poor model when predicting new names
# The MLP (multi-layer perceptron) model proposes that we increase our context
# length to 3, to make more informed guesses for the next character in a sequence

In [105]:
# build the dataset
block_size = 3 # context length
X, Y = [], []
for w in words:
    # print(w)
    # Creating a context array of size (block_size)
    # note that 0 maps to '.' special character
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        # print(f'{''.join(itos[i] for i in context)} -> {ch}')
        context = context[1:] + [ix]

X = torch.tensor(X, device=device)
Y = torch.tensor(Y, device=device)

In [59]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [60]:
# Implementing the embedding lookup table
# The idea is to creating "word vectors" for our context arrays
# We will start by trying to express them in two dimensional space

# lookup table for 27 characters
g = torch.Generator(2147483647)
C = torch.randn((27, 2), generator=g, device=device)

In [61]:
# note that C[5] is equivalent to the one_hot encoding of 5 @ C
# since through matmul, we would only pick out the 5 row of the
# C matrix
# With this in mind, we can simply index into our C matrix

print(F.one_hot(torch.tensor(5), num_classes=27).float().to(device=device) @ C)
print(C[5])

tensor([-0.0011, -0.3644], device='mps:0')
tensor([-0.0011, -0.3644], device='mps:0')


In [62]:
# We can also index with a tensor
# 1D Case
C[torch.tensor([5, 6, 7])]

tensor([[-1.0883e-03, -3.6440e-01],
        [-7.4800e-01,  6.9759e-01],
        [-2.2528e+00, -2.1919e-01]], device='mps:0')

In [63]:
# 2D Case
# Recall X.Shape = [32, 3] with elements ranging from 0-26
# for each row in X, we have 3 indicies which pick out
# an R2 vector from C
C[X].shape

torch.Size([32, 3, 2])

In [64]:
print(X[13, 2])
print(C[X][13, 2])
print(C[1])

tensor(1, device='mps:0')
tensor([1.0474, 0.5927], device='mps:0')
tensor([1.0474, 0.5927], device='mps:0')


In [65]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [66]:
# Creating the fully connected weights from input layer 
# to first hidden layer (6 neurons to 100 neurons)
W1 = torch.randn((6, 100), generator=g, device=device)
# array of biases
b1 = torch.randn(100, generator=g, device=device)

In [67]:
# We want something like emb @ W1 + b1 but emb is a rank 3 tensor
# of the shape (32, 3, 2). We need to flatten the tensor into 
# a (32, 6) tensor

# Here is what we want to happen
# grabbing the embedding for the first character for each example
print(emb[:, 0, :].shape)
# concatenating all the embedding for each character across each example
# along the dim 1 (Note: dims are 0 and 1)
print(torch.cat((emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]), dim=1).shape)

torch.Size([32, 2])
torch.Size([32, 6])


In [68]:
# The code above scales poorly if we wish to change out block size
# so we should use instead the torch.unbind() function, which removes a dimension
# from the tensor, returning a tuple of lower dimension tensors along each
# slice of the tensor in that dimension
# Recall shape (32, 3, 2): torch.unbind(emb, 1) is -> (emb[:, 0, :], emb[:, 1, :], emb[:, 2, :])
# it splits the 3d tensor along dim 1 into 3 separate 2d tensors
# Note: torch.cat is not very efficient because it creates new memory
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [69]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [70]:
a.shape

torch.Size([18])

In [71]:
# we can use the .view() function
# to reshape a tensor with given 
# (2, 9), (3, 3, 2), ...
a.view(2, 9)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [72]:
# In pytorch the .view() is an extremely efficient operation
# this is because under the hood, all tensors are stored 
# as a flattened 1D array, and are interpreted as an
# n dimensional tensor. When we call .view(), we are
# simply telling pytorch how to interpret the stored 
# 1D tensor (storage offset, strides, and shape)
a.storage()

 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [73]:
# We can verify that these two tensors are in fact the same by
# creating a boolean mask and checking if all the elements are true
(emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)).all()

tensor(True, device='mps:0')

In [74]:
# using -1, pytorch will infer the correct shape for
# the view tensor. equivalent to emb.shape[0] in this case
# since we have the (_, 6).
# tanh introduces non-linearity: maps (-inf, inf) -> (-1, 1)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h.shape

torch.Size([32, 100])

In [76]:
W2 = torch.randn(100, 27, generator=g, device=device)
b2 = torch.randn(27, generator=g, device=device)

In [77]:
logits = h @ W2 + b2

In [78]:
logits.shape

torch.Size([32, 27])

In [80]:
counts = logits.exp()
probs = counts / counts.sum(dim=1, keepdim=True)
probs.shape

torch.Size([32, 27])

In [81]:
probs[torch.arange(32), Y]

tensor([2.3252e-07, 3.2908e-06, 7.7671e-05, 5.0160e-09, 8.6244e-01, 2.2144e-04,
        6.5667e-08, 2.2498e-18, 1.2819e-13, 3.9199e-08, 2.4371e-10, 5.8063e-02,
        4.8096e-03, 5.0794e-10, 9.8716e-01, 4.0087e-04, 4.2757e-14, 3.1302e-08,
        9.6571e-03, 1.4320e-08, 8.9676e-02, 2.0804e-03, 4.4685e-07, 7.7768e-06,
        2.5795e-08, 1.7936e-08, 3.9750e-01, 5.4773e-04, 1.3127e-02, 3.8030e-14,
        7.5532e-03, 1.4134e-01], device='mps:0')

In [82]:
probs[torch.arange(32), Y].log()

tensor([-1.5274e+01, -1.2624e+01, -9.4630e+00, -1.9111e+01, -1.4799e-01,
        -8.4154e+00, -1.6539e+01, -4.0636e+01, -2.9685e+01, -1.7055e+01,
        -2.2135e+01, -2.8462e+00, -5.3371e+00, -2.1401e+01, -1.2919e-02,
        -7.8219e+00, -3.0783e+01, -1.7280e+01, -4.6401e+00, -1.8062e+01,
        -2.4115e+00, -6.1752e+00, -1.4621e+01, -1.1764e+01, -1.7473e+01,
        -1.7836e+01, -9.2257e-01, -7.5097e+00, -4.3331e+00, -3.0900e+01,
        -4.8858e+00, -1.9566e+00], device='mps:0')

In [83]:
# Plucking out the probability for each target
# think of it as zipping these two arrays and indexing 
# into a matrix (probs[i, Y[i]] for i in range(32))
loss = -probs[torch.arange(32), Y].log().mean()

In [97]:
loss

tensor(15.0275, device='mps:0')

In [85]:
# Now all together

In [106]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [140]:
# Generators are typically on the cpu, even though we defaulted to
# using mps
g = torch.Generator(device=device).manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [126]:
print(f'The total number of parameters for this model: {sum(p.nelement() for p in parameters)}')

The total number of parameters for this model: 3481


In [ ]:
# These three lines are known as cross entropy
# which we can do using pytorch much more efficiently
# doing in this way is fairly inefficient because we 
# are making new memory for each tensor
# It also makes backward passes more efficient
# counts = logits.exp()
# probs = counts / counts.sum(dim=1, keepdim=True)
# loss = -probs[torch.arange(32), Y].log().mean()

In [141]:
for p in parameters:
    p.requires_grad = True

In [146]:
for i in range(10_000):
    # mini-batch construct
    ix = torch.randint(0, X.shape[0], (32, ))

    # forward pass
    # C[X] for every element in X, pull the embedding in C
    # imagine the matrix being stretched with the embeddings
    # X - (32, 3) -> C(X) - (32, 3, 2)
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        # we can also consider other learning rates
        # as we start reaching the late stages of training
        p.data += -0.01 * p.grad
    

print(loss.item())


2.3553104400634766


In [ ]:
# Notice that the loss are not always decreasing 
# This is because the gradient from the mini-batches are not the exact
# gradient, but an approximation. However, it is significantly faster to compute
# and in practice, it is better to take many small steps then a few big ones

# Secondly, the loss is computed only for that mini-batch, so lets check the loss for
# the full data

In [147]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(2.3470, device='mps:0', grad_fn=<NllLossBackward0>)

In [ ]:
# getting a mini-batch of 32 indexes from the training
# set
torch.randint(0, X.shape[0], (32, ))

tensor([0, 3, 3, 0, 2, 2, 4, 4, 2, 2, 1, 3, 4, 1, 3, 0, 2, 0, 4, 4, 2, 0, 3, 2,
        2, 1, 4, 2, 2, 0, 4, 1], device='mps:0')

In [148]:
# training split, validation split, test split
# 80%, 10%, 10%

# build the data set
def build_dataset(words):
    block_size = 3 # context length
    X, Y = [], []
    for w in words:
        # print(w)
        # Creating a context array of size (block_size)
        # note that 0 maps to '.' special character
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(f'{''.join(itos[i] for i in context)} -> {ch}')
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
n1, n2 - n1, len(words) - n2

(25626, 3203)

In [ ]:
C = torch.randn((27, 2))
